# AI–Quantum Portfolio: xây dựng từ đầu với dữ liệu tải từ máy

**Bản triển khai độc lập — Data NCKH, audit 5/9.** Toàn bộ hàm xử lý, dự báo,
giảm vũ trụ, QUBO, mô phỏng XY-QAOA, phân bổ và backtest được định nghĩa trực tiếp
trong các cell dưới đây. Chỉ các thư viện khoa học tiêu chuẩn được cài qua pip.

**Cách sử dụng trên Colab:** mở notebook → Run all → tại hộp thoại, chọn
`data_5_9.zip` (khoảng 4 MB) hoặc `data_5_9.csv` (khoảng 81 MB) từ máy.
Workbook `Data NCKH - Audit 5_9.xlsx` chỉ chứa 5.000 dòng giá trích xuất,
nên không đủ để chạy nghiên cứu đầy đủ. ZIP/CSV giữ đủ 174.626 dòng PRICE.

Các tham số được công khai tại mục 1. Đây là thực nghiệm viết lại trên bộ dữ liệu
đã biết, **không phải kiểm định xác nhận trên holdout chưa từng được xem**.
Cơ chế huấn luyện, giao dịch và bộ giải được trình bày lại; không kỳ vọng số liệu
bằng tuyệt đối với implementation của báo cáo cũ. Mọi kết quả dưới đây được tính
từ đầu trong lần chạy hiện tại, bao gồm kết quả theo seed.

| Mục | Công việc | Sản phẩm |
|---|---|---|
| 0–3 | Môi trường, cấu hình, upload, audit | Bảng dữ liệu đã kiểm chứng |
| 4–6 | Đặc trưng, lịch walk-forward, XGBoost/EWMA | Dự báo và vũ trụ theo tháng |
| 7–10 | AUR/QAUR, QUBO, XY-QAOA, tỷ trọng | Danh mục có ràng buộc |
| 11–13 | Giao dịch, backtest và so sánh | NAV, chi phí, H1–H5, độ bền seed |
| 14–17 | Biểu đồ, danh mục kế tiếp, audit, xuất file | Kết quả CSV/PNG/JSON/ZIP |

Luồng chung: **PRICE → đặc trưng nhân quả → XGBoost + EWMA → AUR / QAUR →
cùng portfolio QUBO + XY-QAOA → tỷ trọng → khớp lệnh cuối phiên → NAV**.

## 0. Thư viện

Cell này chỉ cài gói khoa học còn thiếu hoặc phiên bản ngoài khoảng hỗ trợ.
Mã nguồn nghiên cứu và dữ liệu đều nằm trong notebook / tệp tải lên từ máy.
Nếu chạy local, có thể cài sẵn các gói để toàn bộ lần chạy không cần mạng.

In [ ]:
import importlib.util
import importlib.metadata
import subprocess
import sys

requirements = {
    "numpy": ("numpy>=2.0,<3", "2.0", "3"),
    "pandas": ("pandas>=2.2,<3", "2.2", "3"),
    "scipy": ("scipy>=1.13,<2", "1.13", "2"),
    "scikit-learn": ("scikit-learn>=1.5,<2", "1.5", "2"),
    "xgboost": ("xgboost>=2.1,<4", "2.1", "4"),
    "matplotlib": ("matplotlib>=3.8,<4", "3.8", "4"),
}
from packaging.version import Version
missing = []
for package, (spec, minimum, maximum) in requirements.items():
    try:
        version = Version(importlib.metadata.version(package))
        if not Version(minimum) <= version < Version(maximum):
            missing.append(spec)
    except importlib.metadata.PackageNotFoundError:
        missing.append(spec)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Thư viện sẵn sàng:", {p: importlib.metadata.version(p) for p in requirements})

## 1. Cấu hình cố định và quy ước nghiên cứu

Dự báo lợi suất 20 phiên; train trượt 24 tháng, validation 3 tháng để chọn số cây,
test theo tháng. Top-K = 8, danh mục = 4, mỗi tỷ trọng trong [5%, 30%].
Ba seed chạy đủ toàn bộ backtest, dùng chung dự báo đã huấn luyện một lần.
Chi phí 25 bps tính trên **tổng giá trị mua + bán**; cash có lợi suất 0.

Market gate dùng trung bình lợi suất trong vũ trụ hợp lệ tại thời điểm quyết định,
tích lũy 30 phiên: âm thì chuyển mục tiêu về cash. Đây là tham số được ấn định,
không quét chọn tham số theo lợi nhuận sau khi xem test.

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime, timezone
from functools import lru_cache
from itertools import combinations
import hashlib
import json
import math
import os
import time
import zipfile
import io

import numpy as np
import pandas as pd
from scipy import stats, optimize, linalg
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

@dataclass(frozen=True)
class ResearchConfig:
    expected_sha256: str = "b0a16d9f8c31a2a5d4e1ba8f00d49b50f112f149d4fae23b3529df085a45ccb2"
    train_months: int = 24
    validation_months: int = 3
    horizon: int = 20
    training_stride: int = 5
    minimum_history: int = 120
    covariance_window: int = 120
    covariance_span: int = 60
    shrinkage: float = 0.10
    candidates: int = 8
    cardinality: int = 4
    lower_weight: float = 0.05
    upper_weight: float = 0.30
    correlation_penalty: float = 0.20
    risk_aversion: float = 0.50
    transaction_cost_bps: float = 25.0
    market_gate_lookback: int = 30
    qa_restarts: int = 4
    qa_steps: int = 160
    qaoa_depth: int = 2
    qaoa_trials: int = 12
    qaoa_shots: int = 512
    seeds: tuple = (42, 43, 44)
    xgb_estimators: int = 160
    bootstrap_samples: int = 1000

CFG = ResearchConfig()
assert CFG.cardinality * CFG.lower_weight <= 1 <= CFG.cardinality * CFG.upper_weight
assert 1 <= CFG.cardinality <= CFG.candidates <= 12
assert CFG.seeds and CFG.transaction_cost_bps >= 0
STARTED = time.perf_counter()
# LOCAL_DATA_PATH giúp chạy cùng notebook trong Jupyter local. Trên Colab để trống.
LOCAL_DATA_PATH = os.environ.get("NCKH_LOCAL_DATA_PATH", "")
BASE = Path.cwd()  # Colab mặc định là /content; Jupyter local dùng thư mục notebook.
RUN_DIR = BASE / ("nckh_rebuild_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN_DIR.mkdir(parents=True, exist_ok=False)
display(pd.DataFrame([asdict(CFG)]).T.rename(columns={0: "giá trị"}))

## 2. Upload ZIP/CSV từ máy và kiểm chứng danh tính

Hàm chỉ đọc tệp đã upload. Với ZIP, đọc thành viên CSV trực tiếp, không bung các
đường dẫn tùy ý. Hash được kiểm tra trước khi parse để tránh chạy nhầm mẫu Excel
hoặc bộ dữ liệu khác. Không thay giá, bổ sung quan sát hoặc lấy kết quả bên ngoài.

In [ ]:
def read_uploaded_dataset(filename, payload):
    """Nhận byte từ CSV/ZIP; trả bảng chuẩn và dấu vết nguồn."""
    suffix = Path(filename).suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        raise ValueError("Excel là bản xem 5.000 dòng. Hãy upload data_5_9.zip hoặc data_5_9.csv đầy đủ.")
    if suffix == ".zip":
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            members = [m for m in archive.infolist() if m.filename.lower().endswith('.csv')]
            if len(members) != 1 or members[0].file_size > 150_000_000:
                raise ValueError("ZIP phải chứa đúng một CSV chuẩn có dung lượng dưới 150 MB.")
            csv_bytes = archive.read(members[0])  # zipfile tự kiểm tra CRC khi đọc.
            member_name = members[0].filename
    elif suffix == ".csv":
        csv_bytes, member_name = payload, Path(filename).name
    else:
        raise ValueError("Định dạng hỗ trợ: .zip hoặc .csv.")
    digest = hashlib.sha256(csv_bytes).hexdigest()
    if digest != CFG.expected_sha256:
        raise ValueError(f"Sai bộ dữ liệu Audit 5/9. SHA-256 nhận được: {digest}")
    data = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)
    provenance = {"uploaded_name": Path(filename).name, "csv_member": member_name,
                  "sha256": digest, "csv_bytes": len(csv_bytes), "loaded_from": "local_upload"}
    return data, provenance

if LOCAL_DATA_PATH:
    upload_path = Path(LOCAL_DATA_PATH)
    raw, provenance = read_uploaded_dataset(upload_path.name, upload_path.read_bytes())
else:
    from google.colab import files
    uploaded = files.upload()
    candidates = [name for name in uploaded if Path(name).suffix.lower() in {".zip", ".csv"}]
    if len(candidates) != 1:
        raise ValueError("Chọn đúng một data_5_9.zip hoặc data_5_9.csv; Excel trích xuất không đủ dữ liệu.")
    name = candidates[0]
    raw, provenance = read_uploaded_dataset(name, uploaded[name])
    del uploaded
print("Đã nhận dữ liệu từ máy:", provenance)

## 3. Audit đầu vào và giới hạn dữ liệu

PRICE có 174.626 quan sát, 120 mã, từ 02/01/2020 đến 28/08/2026. BENCHMARK dừng
tại 31/12/2025; không kéo dài hoặc điền lợi suất benchmark bằng 0 trong 2026.
`available_at` trong dữ liệu là quy ước ngày kế tiếp, không phải bằng chứng lưu trữ
point-in-time độc lập. Panel 120 mã có rủi ro thiên lệch lựa chọn/sống sót.
Giá điều chỉnh có thể được nhà cung cấp tính hồi tố; các kiểm tra thời gian dưới đây
kiểm soát code, không chứng minh nguồn dữ liệu vốn đã hoàn toàn point-in-time.

In [ ]:
EXPECTED_COUNTS = {"PRICE": 174626, "SECURITY": 394, "BENCHMARK": 1499,
                   "CORPORATE_ACTION": 2653, "METADATA": 1}
assert raw.shape == (179173, 64), raw.shape
assert raw.record_type.value_counts().to_dict() == EXPECTED_COUNTS
prices = raw.loc[raw.record_type.eq("PRICE")].copy()
securities = raw.loc[raw.record_type.eq("SECURITY")].copy()
benchmark = raw.loc[raw.record_type.eq("BENCHMARK")].copy()
for frame in (prices, benchmark):
    frame["date"] = pd.to_datetime(frame["date"], format="mixed", errors="raise")
prices["available_at"] = pd.to_datetime(prices["available_at"], format="mixed", errors="raise")
for column in ("listing_date", "delisting_date", "effective_from", "effective_to"):
    securities[column] = pd.to_datetime(securities[column], format="mixed", errors="coerce")
numeric = ["open", "high", "low", "close", "adjusted_close", "volume", "trading_value"]
prices[numeric] = prices[numeric].apply(pd.to_numeric, errors="raise")
assert prices[["date", "ticker", "available_at", *numeric]].notna().all().all()
assert np.isfinite(prices[numeric].to_numpy()).all()
assert (prices.adjusted_close > 0).all()
assert (prices[["volume", "trading_value"]] >= 0).all().all()
assert not prices.duplicated(["ticker", "date"]).any()
assert prices.ticker.nunique() == 120
assert prices.date.min() == pd.Timestamp("2020-01-02")
assert prices.date.max() == pd.Timestamp("2026-08-28")
assert (prices.available_at == prices.date + pd.Timedelta(days=1)).all()
assert not securities.duplicated("ticker").any()
assert set(prices.ticker) <= set(securities.ticker)
data_audit = pd.DataFrame([
    ["CSV SHA-256", provenance["sha256"]], ["Tổng số dòng", len(raw)],
    ["Dòng PRICE", len(prices)], ["Mã PRICE", prices.ticker.nunique()],
    ["Khóa PRICE trùng", int(prices.duplicated(["ticker", "date"]).sum())],
    ["PRICE 2026 (provisional)", int(prices.date.ge("2026-01-01").sum())],
    ["Benchmark kết thúc", str(benchmark.date.max().date())],
], columns=["kiểm tra", "giá trị"])
display(data_audit)

## 4. Panel giá, đặc trưng và nhãn dự báo

Một hàng đặc trưng dùng dữ liệu đến cuối phiên t. Nhãn bắt đầu tại giá đóng cửa
phiên t+1, kết thúc t+21, tương ứng giao dịch sau khi tín hiệu đã khả dụng.
Khi huấn luyện, cả ngày đặc trưng và ngày kết thúc nhãn phải trước mốc cắt.

Giá thiếu được giữ NaN trong bảng quan sát gốc. Riêng sổ định giá dùng giá quan sát
cuối cùng; vị thế không có giao dịch mới được đánh dấu stale. Không backfill giá
trước khi niêm yết. Các chỉ tiêu động lượng/rủi ro dùng cùng quy ước định giá đó.

In [ ]:
prices = prices.sort_values(["date", "ticker"])
DATES = pd.DatetimeIndex(sorted(prices.date.unique()))
TICKERS = sorted(prices.ticker.unique())
observed_prices = prices.pivot(index="date", columns="ticker", values="adjusted_close").reindex(index=DATES, columns=TICKERS)
volume_panel = prices.pivot(index="date", columns="ticker", values="volume").reindex_like(observed_prices)
value_panel = prices.pivot(index="date", columns="ticker", values="trading_value").reindex_like(observed_prices)
OBSERVED = observed_prices.notna()
MARK = observed_prices.ffill()
RETURNS = MARK.pct_change(fill_method=None)
TRADEABLE = OBSERVED & volume_panel.gt(0)
SECURITY = securities.set_index("ticker")

FEATURE_NAMES = ["momentum_5", "momentum_20", "momentum_60", "volatility_20",
                 "volatility_60", "log_liquidity_20", "volume_ratio", "drawdown_60"]
def feature_frame(price, volume, value, observed):
    """Một tài sản: đặc trưng chỉ dựa quá khứ; nhãn nằm ở cột riêng."""
    returns = price.pct_change(fill_method=None)
    table = pd.DataFrame(index=price.index)
    for span in (5, 20, 60):
        table[f"momentum_{span}"] = price.pct_change(span, fill_method=None)
    for span in (20, 60):
        table[f"volatility_{span}"] = returns.rolling(span, min_periods=span).std(ddof=1)
    liquidity = value.fillna(0).rolling(20, min_periods=20).mean()
    table["log_liquidity_20"] = np.log1p(liquidity)
    mean_volume = volume.fillna(0).rolling(20, min_periods=20).mean()
    table["volume_ratio"] = volume / mean_volume.where(mean_volume > 0)
    table["drawdown_60"] = price / price.rolling(60, min_periods=60).max() - 1
    table["liquidity"] = liquidity
    table["history_count"] = observed.astype(int).cumsum()
    table["observed_today"] = observed
    table["volume_today"] = volume
    table["target"] = price.shift(-(CFG.horizon + 1)) / price.shift(-1) - 1
    table["label_end"] = pd.Series(price.index, index=price.index).shift(-(CFG.horizon + 1))
    table["available_at"] = table.index + pd.Timedelta(days=1)
    return table.replace([np.inf, -np.inf], np.nan)

features = pd.concat([
    feature_frame(MARK[t], volume_panel[t], value_panel[t], OBSERVED[t]).assign(ticker=t)
    for t in TICKERS
]).rename_axis("date").reset_index().sort_values(["date", "ticker"]).reset_index(drop=True)
features["sample_training"] = features.date.isin(DATES[::CFG.training_stride])
print("Bảng đặc trưng:", features.shape, "; đặc trưng:", FEATURE_NAMES)
display(features[["date", "ticker", *FEATURE_NAMES, "target", "label_end"]].dropna().head(6))

## 5. Lịch walk-forward và quy tắc thời gian

Quyết định vào 00:00 phiên đầu tháng, dùng giá phiên trước. Mua/bán tại giá cuối
phiên quyết định. Mô hình không hưởng lợi suất từ phiên trước đến phiên khớp lệnh
đối với cổ phiếu mới mua. Train/validation loại các nhãn vươn qua mốc cắt.
Nhãn test chỉ dùng để đánh giá sau dự báo. Cuối mẫu có thêm một quyết định dự kiến
cho tháng kế tiếp, không đưa vào lợi nhuận backtest.

In [ ]:
def make_calendar(dates):
    first_month = dates.min().to_period("M").to_timestamp()
    start = first_month + pd.DateOffset(months=CFG.train_months + CFG.validation_months)
    decisions = []
    for month in pd.period_range(start, dates.max(), freq="M"):
        sessions = dates[dates.to_period("M") == month]
        if len(sessions) == 0:
            continue
        execution = sessions[0]
        signal = dates[dates < execution][-1]
        valid_start = execution - pd.DateOffset(months=CFG.validation_months)
        decisions.append({"fold": len(decisions), "decision": execution, "signal_date": signal,
                          "train_start": valid_start - pd.DateOffset(months=CFG.train_months),
                          "validation_start": valid_start, "test_end": sessions[-1], "prospective": False})
    next_month = (dates.max().to_period("M") + 1).to_timestamp()
    valid_start = next_month - pd.DateOffset(months=CFG.validation_months)
    decisions.append({"fold": len(decisions), "decision": next_month, "signal_date": dates[-1],
                      "train_start": valid_start - pd.DateOffset(months=CFG.train_months),
                      "validation_start": valid_start, "test_end": pd.NaT, "prospective": True})
    return pd.DataFrame(decisions)

calendar = make_calendar(DATES)
assert (calendar.train_start < calendar.validation_start).all()
assert (calendar.validation_start < calendar.signal_date).all()
assert (calendar.signal_date < calendar.decision).all()
display(calendar.head())
print("Số tháng backtest:", int((~calendar.prospective).sum()), "; quyết định kế tiếp:", calendar.iloc[-1].decision.date())

## 6. Dự báo XGBoost và EWMA covariance dùng chung

XGBoost chọn số cây bằng early stopping trên validation đã purge, sau đó fit lại
trên các nhãn đã hoàn tất trước quyết định. Medians bù đặc trưng thiếu chỉ fit trên
tập train tương ứng. Mỗi tháng, một bộ dự báo/covariance được chia sẻ cho AUR và QAUR.
Rủi ro dùng trọng số EWMA và shrinkage về đường chéo, đảm bảo ma trận PSD.
Vũ trụ chỉ gồm mã đã niêm yết, có tối thiểu 120 quan sát, có giá và khối lượng ở
phiên tín hiệu. Không yêu cầu phải có dữ liệu trong tương lai mới được chọn.

In [ ]:
def ewma_covariance(history, span, shrinkage):
    x = history.to_numpy(float)
    if not np.isfinite(x).all() or len(x) < 30:
        raise ValueError("Lịch sử covariance thiếu dữ liệu hoặc quá ngắn.")
    age = np.arange(len(x) - 1, -1, -1)
    weights = (1 - 2 / (span + 1)) ** age
    weights /= weights.sum()
    centered = x - weights @ x
    cov = (centered.T * weights) @ centered / (1 - weights @ weights)
    cov = (1 - shrinkage) * cov + shrinkage * np.diag(np.diag(cov))
    cov += np.eye(cov.shape[0]) * 1e-10
    return (cov + cov.T) / 2

def finite_matrix(frame, medians=None):
    x = frame[FEATURE_NAMES].replace([np.inf, -np.inf], np.nan)
    if medians is None:
        medians = x.median()
        if medians.isna().any():
            raise ValueError("Có đặc trưng không có quan sát train hợp lệ.")
    return x.fillna(medians).astype(np.float32), medians

def new_regressor(trees, stopping=None):
    return XGBRegressor(n_estimators=int(trees), max_depth=3, learning_rate=0.05,
                        min_child_weight=20, subsample=1.0, colsample_bytree=1.0,
                        reg_lambda=5.0, objective="reg:squarederror", tree_method="hist",
                        n_jobs=2, random_state=CFG.seeds[0], early_stopping_rounds=stopping)

def forecast_one_fold(fold, all_features):
    sampled = all_features.loc[all_features.sample_training & all_features.observed_today
                               & all_features.target.notna()].copy()
    sampled = sampled.loc[sampled.history_count >= CFG.minimum_history]
    train = sampled.loc[(sampled.date >= fold.train_start)
                        & (sampled.label_end < fold.validation_start)]
    valid = sampled.loc[(sampled.date >= fold.validation_start)
                        & (sampled.label_end < fold.decision)]
    fit = sampled.loc[(sampled.date >= fold.train_start) & (sampled.label_end < fold.decision)]
    assert len(train) > 1000 and len(valid) > 100
    assert train.available_at.max() < fold.validation_start
    assert valid.available_at.max() < fold.decision
    x_train, medians = finite_matrix(train)
    x_valid, _ = finite_matrix(valid, medians)
    probe = new_regressor(CFG.xgb_estimators, stopping=15)
    probe.fit(x_train, train.target, eval_set=[(x_valid, valid.target)], verbose=False)
    val_prediction = probe.predict(x_valid)
    rank_ic = stats.spearmanr(val_prediction, valid.target).statistic
    trees = probe.best_iteration + 1
    x_fit, final_medians = finite_matrix(fit)
    model = new_regressor(trees)
    model.fit(x_fit, fit.target, verbose=False)
    snap = all_features.loc[all_features.date.eq(fold.signal_date)].copy()
    snap = snap.loc[snap.observed_today & snap.volume_today.gt(0)
                    & snap.liquidity.gt(0) & snap.history_count.ge(CFG.minimum_history)
                    & snap.available_at.le(fold.decision)]
    listed = SECURITY.reindex(snap.ticker)
    permitted = ((listed.listing_date.isna() | listed.listing_date.le(fold.decision))
                 & (listed.delisting_date.isna() | listed.delisting_date.gt(fold.decision)))
    snap = snap.loc[permitted.to_numpy()].sort_values("ticker")
    history = RETURNS.loc[:fold.signal_date, snap.ticker].tail(CFG.covariance_window)
    complete = history.columns[history.notna().all()]
    snap = snap.loc[snap.ticker.isin(complete)].copy()
    assert len(snap) >= CFG.candidates
    x_snap, _ = finite_matrix(snap, final_medians)
    snap["prediction_20d"] = model.predict(x_snap)
    snap["ewma_daily_return"] = history[snap.ticker].ewm(span=CFG.covariance_span).mean().iloc[-1].to_numpy()
    # Hai nhánh dùng cùng dự báo XGBoost; EWMA mô tả lợi suất lịch sử.
    score = (0.70 * snap.prediction_20d.rank(pct=True)
             + 0.20 * snap.liquidity.rank(pct=True)
             + 0.10 * (-snap.volatility_20).rank(pct=True))
    snap["common_score"] = score
    hist = history[snap.ticker]
    covariance = ewma_covariance(hist, CFG.covariance_span, CFG.shrinkage)
    scale = np.sqrt(np.diag(covariance))
    correlation = np.clip(covariance / np.outer(scale, scale), -1, 1)
    market = hist.mean(axis=1).tail(CFG.market_gate_lookback)
    growth = float(np.prod(1 + market) - 1)
    diagnostic = {"fold": fold.fold, "decision": fold.decision, "signal_date": fold.signal_date,
                  "train_rows": len(train), "validation_rows": len(valid), "refit_rows": len(fit),
                  "train_label_end": train.label_end.max(), "validation_label_end": valid.label_end.max(),
                  "refit_label_end": fit.label_end.max(), "validation_start": fold.validation_start,
                  "available_at_max": snap.available_at.max(), "trees": trees,
                  "validation_rank_ic": float(rank_ic), "universe_size": len(snap),
                  "market_growth": growth, "gate_exposure": float(growth >= 0)}
    return {"fold": fold, "snapshot": snap.reset_index(drop=True), "covariance": covariance,
            "correlation": np.abs(correlation), "diagnostic": diagnostic}

In [ ]:
forecast_cache = []
for fold in calendar.itertuples(index=False):
    forecast_cache.append(forecast_one_fold(fold, features))
    print(f"Dự báo {fold.fold + 1}/{len(calendar)}: {fold.decision.date()}, "
          f"universe={forecast_cache[-1]['diagnostic']['universe_size']}", flush=True)
forecast_diagnostics = pd.DataFrame([item["diagnostic"] for item in forecast_cache])
forecast_snapshots = pd.concat([item["snapshot"].assign(fold=item["fold"].fold,
                                                      decision=item["fold"].decision)
                                for item in forecast_cache], ignore_index=True)
display(forecast_diagnostics.tail())

## 7. AUR và QAUR: cùng đầu vào, khác phép chọn tập con

AUR chọn K mã có common_score cao nhất. QAUR tối đa hóa
$F(S)=\frac{1}{K}\sum_{i\in S}s_i-\lambda\,\mathrm{mean}_{i<j\in S}|\rho_{ij}|$
bằng simulated annealing với phép đổi một mã vào/ra giữ nguyên K.
Đây là **bộ giải cổ điển cho bài toán QUBO**, không phải phần cứng lượng tử.
Khởi tạo bằng AUR nên objective tốt hơn/không kém AUR là tính chất thiết kế;
không được diễn giải riêng kết quả đó thành bằng chứng lợi thế lượng tử.

In [ ]:
def reduction_value(indices, scores, correlation):
    block = correlation[np.ix_(indices, indices)]
    avg_corr = float(block[np.triu_indices(len(indices), 1)].mean())
    return float(scores[indices].mean() - CFG.correlation_penalty * avg_corr)

def reduce_assets(method, scores, correlation, seed):
    k, n = CFG.candidates, len(scores)
    initial = np.argsort(-scores, kind="stable")[:k]
    if method == "AUR":
        return np.sort(initial)
    if method != "QAUR":
        raise ValueError(method)
    rng = np.random.default_rng(seed)
    best = initial.copy()
    best_value = reduction_value(best, scores, correlation)
    for restart in range(CFG.qa_restarts):
        current = initial.copy() if restart == 0 else rng.choice(n, k, replace=False)
        current_value = reduction_value(current, scores, correlation)
        for step in range(CFG.qa_steps):
            outside = np.setdiff1d(np.arange(n), current)
            if not len(outside):
                break
            candidate = current.copy()
            candidate[rng.integers(k)] = rng.choice(outside)
            value = reduction_value(candidate, scores, correlation)
            temperature = 0.02 * (0.001 / 0.02) ** (step / max(CFG.qa_steps - 1, 1))
            if value >= current_value or rng.random() < np.exp((value - current_value) / temperature):
                current, current_value = candidate, value
            if current_value > best_value:
                best, best_value = current.copy(), current_value
    return np.sort(best)

## 8. Portfolio QUBO và mô phỏng XY-QAOA độc lập

Sau Top-K, hai nhánh dùng cùng ma trận $Q$ và bộ giải. Với $z\in\{0,1\}^K$,
$\sum z_i=k$, năng lượng là $z^TQz$: covariance được chuẩn hóa theo phương sai
điển hình, kỳ vọng lợi suất theo độ lệch chuẩn chéo tài sản.

Bộ mô phỏng biểu diễn trực tiếp $\binom{8}{4}=70$ trạng thái khả thi. Mixer nối hai
trạng thái khác nhau bởi một phép đổi 1↔0 (full XY mixer; hệ số được hấp thụ vào beta).
Bắt đầu từ trạng thái Dicke đồng đều, xen kẽ phase chi phí và mixer p lần, chọn góc
bằng ngân sách tìm kiếm công khai, lấy mẫu shots. Danh mục là mẫu có năng lượng thấp
nhất đã đo; nghiệm duyệt toàn bộ chỉ dùng để báo khoảng cách tới tối ưu.

In [ ]:
def portfolio_qubo(expected_return, covariance, cardinality, risk_aversion):
    mu = np.asarray(expected_return, dtype=float)
    cov = np.asarray(covariance, dtype=float)
    mu_scale = max(float(np.std(mu)), 1e-6)
    risk_scale = max(float(np.median(np.diag(cov))), 1e-10)
    return risk_aversion * cov / risk_scale / cardinality**2 - np.diag(mu / mu_scale) / cardinality

@lru_cache(maxsize=8)
def feasible_mixer(n, k):
    basis = np.zeros((math.comb(n, k), n), dtype=float)
    for row, selected in enumerate(combinations(range(n), k)):
        basis[row, list(selected)] = 1
    adjacency = (np.sum(np.abs(basis[:, None, :] - basis[None, :, :]), axis=2) == 2).astype(float)
    eigenvalues, eigenvectors = linalg.eigh(adjacency)
    return basis, eigenvalues, eigenvectors

def xy_qaoa(q, k, seed):
    n = len(q)
    basis, eigenvalues, eigenvectors = feasible_mixer(n, k)
    energies = np.einsum("bi,ij,bj->b", basis, q, basis)
    scaled = (energies - energies.min()) / max(float(np.std(energies)), 1e-10)
    initial = np.full(len(basis), 1 / np.sqrt(len(basis)), dtype=complex)
    def probabilities(angles):
        state = initial.copy()
        for layer in range(CFG.qaoa_depth):
            gamma, beta = angles[2 * layer:2 * layer + 2]
            state *= np.exp(-1j * gamma * scaled)
            state = eigenvectors @ (np.exp(-1j * beta * eigenvalues) * (eigenvectors.T @ state))
        probs = np.abs(state)**2
        assert abs(probs.sum() - 1) < 1e-8
        return probs / probs.sum()
    rng = np.random.default_rng(seed)
    options = np.vstack([np.zeros(2 * CFG.qaoa_depth), rng.uniform(-np.pi, np.pi, (CFG.qaoa_trials, 2 * CFG.qaoa_depth))])
    expectations = [float(probabilities(angle) @ energies) for angle in options]
    best_angles = options[int(np.argmin(expectations))].copy()
    best_energy = min(expectations)
    for step_size in (0.6, 0.2, 0.06):
        for coordinate in range(len(best_angles)):
            for sign in (-1, 1):
                trial = best_angles.copy()
                trial[coordinate] += sign * step_size
                value = float(probabilities(trial) @ energies)
                if value < best_energy:
                    best_angles, best_energy = trial, value
    probs = probabilities(best_angles)
    measured = rng.choice(len(basis), size=CFG.qaoa_shots, p=probs)
    winner = measured[np.argmin(energies[measured])]
    optimal = float(energies.min())
    return {"selected": np.flatnonzero(basis[winner]), "energy": float(energies[winner]),
            "exact_energy": optimal, "optimality_gap": float(energies[winner] - optimal),
            "expected_energy": float(probs @ energies),
            "optimal_probability": float(probs[np.isclose(energies, optimal, atol=1e-9)].sum()),
            "feasibility_rate": float(np.all(basis[measured].sum(axis=1) == k)),
            "probability_sum": float(probs.sum()), "angles": best_angles.tolist(), "states": len(basis)}

## 9. Tỷ trọng bị chặn trên simplex

Trọng số nền nghịch đảo volatility được chiếu Euclid lên tổng bằng 1,
với từng trọng số trong [lower, upper]. Bisection giải biến Lagrange, không lặp
clip-normalize có thể phá ràng buộc. Gate nhân đồng đều vào hai nhánh sau phân bổ.

In [ ]:
def bounded_weights(volatility, lower, upper):
    vol = np.asarray(volatility, dtype=float)
    if not np.isfinite(vol).all() or (vol <= 0).any():
        raise ValueError("Volatility phải hữu hạn và dương.")
    if not len(vol) * lower <= 1 <= len(vol) * upper:
        raise ValueError("Ràng buộc trọng số vô nghiệm.")
    base = 1 / vol
    base /= base.sum()
    lo, hi = float(np.min(base - upper) - 1), float(np.max(base - lower) + 1)
    for _ in range(100):
        middle = (lo + hi) / 2
        weights = np.clip(base - middle, lower, upper)
        if weights.sum() > 1:
            lo = middle
        else:
            hi = middle
    weights = np.clip(base - (lo + hi) / 2, lower, upper)
    assert np.isclose(weights.sum(), 1, atol=1e-10)
    return weights

def build_targets(cache, seed):
    selections, diagnostics, targets = [], [], {}
    for item in cache:
        fold, snap = item["fold"], item["snapshot"]
        score = snap.common_score.to_numpy(float)
        names = snap.ticker.to_numpy()
        cov, corr = item["covariance"], item["correlation"]
        for method in ("AUR", "QAUR"):
            candidate = reduce_assets(method, score, corr, seed + fold.fold * 1009)
            selected_cov = cov[np.ix_(candidate, candidate)]
            q = portfolio_qubo(snap.prediction_20d.to_numpy()[candidate], selected_cov,
                               CFG.cardinality, CFG.risk_aversion)
            solved = xy_qaoa(q, CFG.cardinality, seed + fold.fold * 1009)
            chosen = candidate[solved["selected"]]
            weights = bounded_weights(np.sqrt(np.diag(cov)[chosen]), CFG.lower_weight, CFG.upper_weight)
            exposure = item["diagnostic"]["gate_exposure"]
            target = dict(zip(names[chosen], weights * exposure))
            targets[(fold.decision, method)] = target
            corr_block = corr[np.ix_(candidate, candidate)]
            diagnostics.append({"seed": seed, "fold": fold.fold, "decision": fold.decision,
                                "method": method, "prospective": fold.prospective,
                                "reduction_objective": reduction_value(candidate, score, corr),
                                "candidate_correlation": float(corr_block[np.triu_indices(len(candidate), 1)].mean()),
                                "gate_exposure": exposure,
                                **{key: value for key, value in solved.items() if key not in {"selected", "angles"}}})
            for idx in candidate:
                match = np.flatnonzero(chosen == idx)
                weight = float(weights[match[0]]) if len(match) else 0.0
                selections.append({"seed": seed, "fold": fold.fold, "decision": fold.decision,
                                   "method": method, "ticker": names[idx], "prospective": fold.prospective,
                                   "prediction_20d": float(snap.prediction_20d.iloc[idx]),
                                   "common_score": float(score[idx]), "selected": bool(len(match)),
                                   "shadow_weight": weight, "target_weight": weight * exposure,
                                   "cash_target": 1 - exposure})
        # Baseline tham chiếu đầu tư đều, không gate; không dùng để quy lợi thế cho reducer.
        targets[(fold.decision, "UNIVERSE_EW")] = {t: 1 / len(names) for t in names}
    return pd.DataFrame(selections), pd.DataFrame(diagnostics), targets

## 10. Kiểm chứng toán học trước backtest

Kiểm tra nghiệm QUBO so với phép tính trực tiếp, tổng xác suất, cardinality,
giới hạn tỷ trọng và bảo toàn dữ liệu quá khứ khi giá tương lai bị thay đổi.
Đây là kiểm tra triển khai, không kiểm định giả thuyết tài chính.

In [ ]:
def kernel_checks():
    test_q = np.array([[0.3, 0.1, -0.2, 0.0], [0.1, -0.4, 0.05, 0.0],
                       [-0.2, 0.05, 0.1, 0.1], [0.0, 0.0, 0.1, -0.2]])
    solved = xy_qaoa(test_q, 2, 101)
    direct = min(sum(test_q[i, j] for i in pair for j in pair) for pair in combinations(range(4), 2))
    assert np.isclose(solved["exact_energy"], direct)
    assert solved["feasibility_rate"] == 1 and solved["optimality_gap"] >= -1e-10
    w = bounded_weights([0.01, 0.1, 0.3, 0.5], 0.05, 0.3)
    assert np.isclose(w.sum(), 1) and np.all(w >= 0.05) and np.all(w <= 0.3)
    ticker = TICKERS[-1]
    cut = len(DATES) // 2
    altered = MARK[ticker].copy()
    altered.iloc[cut + 1:] *= 10
    before = feature_frame(MARK[ticker], volume_panel[ticker], value_panel[ticker], OBSERVED[ticker])
    after = feature_frame(altered, volume_panel[ticker], value_panel[ticker], OBSERVED[ticker])
    pd.testing.assert_frame_equal(before.iloc[:cut + 1][FEATURE_NAMES], after.iloc[:cut + 1][FEATURE_NAMES])
    for invalid in ([0, 0.1], [np.nan, 0.2]):
        try:
            bounded_weights(invalid, 0, 1)
        except ValueError:
            continue
        raise AssertionError("Volatility không hợp lệ phải bị từ chối.")
    return True

assert kernel_checks()
print("Các kiểm tra QUBO, XY-QAOA, simplex và đặc trưng nhân quả đã đạt.")

## 11. Sổ giao dịch và backtest tự tài trợ

Mỗi ngày: định giá vị thế cũ → nhận mục tiêu nếu đầu tháng → khớp tại close →
trừ chi phí → cập nhật cash. Giá trị tài sản thay đổi theo lợi suất thực,
tỷ trọng trôi giữa hai kỳ, không giả định tái cân bằng miễn phí hàng ngày.

Với NAV trước lệnh V, giá trị vị thế hiện tại h, mục tiêu w và phí c:
giải $V'=V-c\sum_i|w_iV'-h_i|$. Cash = $V'(1-\sum_iw_i)$.
Mã không có giá/volume tại phiên khớp giữ nguyên vị thế cũ. Các lệnh có thể khớp
vẫn thực hiện; tiền dành cho lệnh mua bị chặn ở lại cash. Nếu vị thế không bán được
chiếm ngân sách, thu nhỏ đồng đều phần mục tiêu khả thi để không vay tiền.
Ledger ghi số lệnh bị chặn và trạng thái khớp một phần. Mục tiêu cardinality/trọng số
được kiểm tra trước khớp; tỷ trọng thực tế có thể lệch do lệnh bị chặn và giá trôi.
Giá thiếu ở vị thế đang giữ được định giá theo lần quan sát gần nhất và đếm stale.

In [ ]:
def execute_rebalance(holdings, cash, target, tradeable, cost_rate):
    holdings = np.asarray(holdings, float)
    target = np.asarray(target, float)
    value = float(holdings.sum() + cash)
    assert value > 0 and target.min() >= -1e-12 and target.sum() <= 1 + 1e-10
    desired_change = target * value - holdings
    can_trade = np.asarray(tradeable, bool)
    blocked = int(np.sum((np.abs(desired_change) > 1e-12) & ~can_trade))
    locked = float(holdings[~can_trade].sum())
    def feasible_target(after_cost):
        allocation = holdings.copy()
        desired = target[can_trade] * after_cost
        budget = max(0, after_cost - locked)
        if desired.sum() > budget and desired.sum() > 0:
            desired *= budget / desired.sum()
        allocation[can_trade] = desired
        return allocation
    def residual(after_cost):
        return after_cost + cost_rate * np.abs(feasible_target(after_cost) - holdings).sum() - value
    after = value if abs(value - locked) < 1e-14 else optimize.brentq(residual, locked, value, xtol=1e-14)
    new_holdings = feasible_target(after)
    new_cash = max(0.0, after - float(new_holdings.sum()))
    traded = float(np.abs(new_holdings - holdings).sum())
    fee = traded * cost_rate
    error = abs(new_holdings.sum() + new_cash + fee - value)
    assert error < 1e-9 and new_cash >= -1e-12
    status = "partial_fill_untradeable" if blocked and traded > 1e-12 else "deferred_missing_trade" if blocked else "executed"
    return new_holdings, new_cash, {"status": status, "blocked_orders": blocked, "cost": fee,
                                   "turnover": traded / value, "reconciliation_error": error}

def simulate(targets, seed):
    dates = DATES[DATES >= calendar.iloc[0].decision]
    return_rows, trade_rows = [], []
    raw_returns = RETURNS.reindex(dates).to_numpy(float)
    observed = OBSERVED.reindex(dates).to_numpy(bool)
    tradeable = TRADEABLE.reindex(dates).to_numpy(bool)
    for method in ("AUR", "QAUR", "UNIVERSE_EW"):
        holdings = np.zeros(len(TICKERS))
        cash, previous_value = 1.0, 1.0
        for row, date in enumerate(dates):
            daily = raw_returns[row]
            if np.any((holdings > 1e-12) & ~np.isfinite(daily)):
                raise ValueError("Thiếu lợi suất định giá cho vị thế đang giữ.")
            holdings *= 1 + np.nan_to_num(daily, nan=0.0)
            before_trade = float(holdings.sum() + cash)
            fee, turnover = 0.0, 0.0
            target_dict = targets.get((date, method))
            if target_dict is not None:
                target = np.array([target_dict.get(t, 0.0) for t in TICKERS])
                holdings, cash, trade = execute_rebalance(holdings, cash, target, tradeable[row],
                                                          CFG.transaction_cost_bps / 10000)
                fee, turnover = trade["cost"], trade["turnover"]
                trade_rows.append({"seed": seed, "method": method, "date": date,
                                   "pretrade_nav": before_trade, **trade})
            value = float(holdings.sum() + cash)
            return_rows.append({"seed": seed, "method": method, "date": date,
                                "net_return": value / previous_value - 1,
                                "gross_return": before_trade / previous_value - 1,
                                "nav": value, "cost": fee, "turnover": turnover,
                                "cash_weight": cash / value,
                                "positions": int((holdings > 1e-12).sum()),
                                "stale_positions": int(((holdings > 1e-12) & ~observed[row]).sum())})
            previous_value = value
    return pd.DataFrame(return_rows), pd.DataFrame(trade_rows)

In [ ]:
# Kiểm tra độc lập chi phí vào lệnh và tỷ trọng trôi theo biến động giá.
h, c, order = execute_rebalance(np.zeros(2), 1, np.array([0.6, 0.4]), [True, True], 0.0025)
assert np.isclose(h.sum(), 1 / 1.0025) and c == 0
drifted = h * [1.1, 0.9]
assert not np.allclose(drifted / drifted.sum(), [0.6, 0.4])
_, _, cancelled = execute_rebalance(h, c, np.array([0.0, 0.0]), [False, True], 0.0025)
assert cancelled["status"] == "partial_fill_untradeable"

selection_runs, solver_runs, return_runs, trade_runs, target_runs = [], [], [], [], {}
for seed in CFG.seeds:
    print(f"Seed {seed}: AUR, QAUR và XY-QAOA trên mọi tháng...", flush=True)
    selection, solver, targets = build_targets(forecast_cache, seed)
    returns, trades = simulate(targets, seed)
    selection_runs.append(selection)
    solver_runs.append(solver)
    return_runs.append(returns)
    trade_runs.append(trades)
    target_runs[seed] = targets
    print(f"Seed {seed}: hoàn thành {returns.date.nunique()} phiên.", flush=True)
selections = pd.concat(selection_runs, ignore_index=True)
solver_results = pd.concat(solver_runs, ignore_index=True)
portfolio_returns = pd.concat(return_runs, ignore_index=True)
trade_ledger = pd.concat(trade_runs, ignore_index=True)
print("Tái cân bằng theo trạng thái:")
display(trade_ledger.groupby(["method", "status"]).size().rename("số lần").to_frame())

## 12. Lợi nhuận, rủi ro và benchmark đúng phạm vi

Sharpe = mean(r)/std(r) × sqrt(252), risk-free = 0. CAGR = tích lũy quy năm;
chúng là hai đại lượng khác nhau. Max drawdown lấy mốc NAV ban đầu bằng 1,
bao gồm lỗ/chi phí ngày đầu. Benchmark VNALLSHARETRI chỉ so sánh trên các ngày
có quan sát thật chung, và là chuỗi chỉ số chưa trừ chi phí thực thi.
Các giai đoạn lịch sử/2026 mô tả thời gian quan sát, không mang nhãn holdout mới.

In [ ]:
def metrics(values):
    r = np.asarray(values, dtype=float)
    if not len(r) or not np.isfinite(r).all() or np.any(r <= -1):
        raise ValueError("Lợi suất đầu vào không hợp lệ.")
    wealth = np.r_[1.0, np.cumprod(1 + r)]
    volatility = float(np.std(r, ddof=1)) if len(r) > 1 else np.nan
    downside = float(np.sqrt(np.mean(np.minimum(r, 0)**2)))
    return {"observations": len(r), "cumulative_return": float(wealth[-1] - 1),
            "annualized_return": float(wealth[-1] ** (252 / len(r)) - 1),
            "annualized_volatility": volatility * np.sqrt(252),
            "sharpe_zero_rf": float(r.mean() / volatility * np.sqrt(252)) if volatility > 0 else np.nan,
            "sortino_zero_rf": float(r.mean() / downside * np.sqrt(252)) if downside > 0 else np.nan,
            "maximum_drawdown": float(np.min(wealth / np.maximum.accumulate(wealth) - 1))}

metric_rows = []
period_bounds = {"all_observed": ("2020-01-01", "2026-12-31"),
                 "historical_to_2025": ("2020-01-01", "2025-12-31"),
                 "observed_2026_provisional": ("2026-01-01", "2026-12-31")}
for (seed, method), block in portfolio_returns.groupby(["seed", "method"]):
    for period, (start, end) in period_bounds.items():
        sample = block.loc[block.date.between(start, end)]
        if len(sample):
            metric_rows.append({"seed": seed, "method": method, "period": period, **metrics(sample.net_return)})
performance = pd.DataFrame(metric_rows)
benchmark_series = benchmark.sort_values("date").set_index("date").total_return_index.pct_change(fill_method=None).dropna()
main = portfolio_returns.loc[portfolio_returns.seed.eq(CFG.seeds[0])]
wide = main.pivot(index="date", columns="method", values="net_return")
common_dates = wide.index.intersection(benchmark_series.index)
benchmark_comparison = pd.DataFrame([
    {"method": method, "start": common_dates.min(), "end": common_dates.max(),
     **metrics(wide.loc[common_dates, method])} for method in wide.columns
] + [{"method": "VNALLSHARETRI_index", "start": common_dates.min(), "end": common_dates.max(),
       **metrics(benchmark_series.loc[common_dates])}])
assert len(common_dates) > 200
display(performance.loc[performance.seed.eq(CFG.seeds[0])].round(5))
display(benchmark_comparison.round(5))

## 13. So sánh reducer, H1–H5 và độ bền qua seed

H1: chênh objective QAUR−AUR > 0; H2: tương quan QAUR thấp hơn; H3: turnover
QAUR−AUR không vượt biên 0,02; H4: trung bình theo tháng của lợi suất ngày QAUR cao hơn.
Các p-value dùng bootstrap theo block tháng để giữ phụ thuộc thời gian trong tháng.
Khoảng tin cậy tái lấy mẫu các tháng liên tiếp (block ba tháng). Holm hiệu chỉnh
bốn kiểm định. H5 báo dấu chênh lợi suất cho mọi seed, không coi các seed là các
mẫu thị trường độc lập. Tất cả là phân tích mô tả trên dữ liệu đã biết.

In [ ]:
def month_block_test(values, boundary=0.0, alternative="greater", seed=42):
    """Bootstrap trung bình theo block ba tháng; kiểm định null được center tại biên."""
    x = np.asarray(values, dtype=float)
    assert np.isfinite(x).all() and len(x) >= 10
    rng = np.random.default_rng(seed)
    block = 3
    starts = rng.integers(0, len(x), (CFG.bootstrap_samples, math.ceil(len(x) / block)))
    indices = ((starts[..., None] + np.arange(block)) % len(x)).reshape(CFG.bootstrap_samples, -1)[:, :len(x)]
    means = x[indices].mean(axis=1)
    estimate = float(x.mean())
    delta = estimate - boundary
    null = means - estimate
    exceed = null >= delta if alternative == "greater" else null <= delta
    pvalue = float((exceed.sum() + 1) / (len(null) + 1))
    return estimate, float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975)), pvalue

primary_solver = solver_results.loc[solver_results.seed.eq(CFG.seeds[0]) & ~solver_results.prospective]
paired = primary_solver.pivot(index="fold", columns="method", values=["reduction_objective", "candidate_correlation"])
comparison_rows = []
for fold, group in selections.loc[selections.seed.eq(CFG.seeds[0]) & ~selections.prospective].groupby("fold"):
    a = set(group.loc[group.method.eq("AUR"), "ticker"])
    q = set(group.loc[group.method.eq("QAUR"), "ticker"])
    comparison_rows.append({"fold": fold, "overlap": len(a & q), "jaccard": len(a & q) / len(a | q)})
reduction_comparison = pd.DataFrame(comparison_rows)
turnover = main.assign(month=main.date.dt.to_period("M")).groupby(["month", "method"]).turnover.sum().unstack()
monthly_daily_difference = (wide.QAUR - wide.AUR).groupby(wide.index.to_period("M")).mean()
tests = [
    ("H1_objective", paired["reduction_objective"].QAUR - paired["reduction_objective"].AUR, 0, "greater"),
    ("H2_correlation", paired["candidate_correlation"].QAUR - paired["candidate_correlation"].AUR, 0, "less"),
    ("H3_turnover_margin", turnover.QAUR - turnover.AUR, 0.02, "less"),
    ("H4_daily_return", monthly_daily_difference, 0, "greater"),
]
hypotheses = pd.DataFrame([
    {"hypothesis": name, "boundary": boundary, "alternative": alternative,
     **dict(zip(["estimate", "ci_2_5pct", "ci_97_5pct", "pvalue"],
                month_block_test(values, boundary, alternative, CFG.seeds[0])))}
    for name, values, boundary, alternative in tests
])
order = np.argsort(hypotheses.pvalue.to_numpy())
adjusted = np.minimum(1, np.maximum.accumulate(hypotheses.pvalue.to_numpy()[order] * np.arange(4, 0, -1)))
hypotheses["holm_pvalue"] = np.nan
hypotheses.loc[order, "holm_pvalue"] = adjusted
hypotheses["supported_holm_5pct"] = hypotheses.holm_pvalue.lt(0.05)
seed_robustness = performance.loc[performance.period.eq("all_observed")].pivot(index="seed", columns="method", values="cumulative_return")
seed_robustness["QAUR_minus_AUR"] = seed_robustness.QAUR - seed_robustness.AUR
seed_robustness["positive_difference"] = seed_robustness.QAUR_minus_AUR.gt(0)
print("H5 — số seed QAUR có lợi nhuận tích lũy cao hơn AUR:", int(seed_robustness.positive_difference.sum()), "/", len(CFG.seeds))
display(hypotheses.round(6))
display(seed_robustness.round(6))

## 14. Biểu đồ từ kết quả vừa chạy

NAV bao gồm phí. Biểu đồ drawdown có gốc vốn bằng 1. Các hình PNG được lưu
trong thư mục kết quả và xuất vào ZIP để đưa vào phần thực nghiệm của báo cáo mới.

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
nav = main.pivot(index="date", columns="method", values="nav")
colors = {"AUR": "#245C91", "QAUR": "#D46A30", "UNIVERSE_EW": "#737B83"}
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for method in nav:
    axes[0, 0].plot(nav.index, nav[method], label=method, color=colors[method])
    running_max = np.maximum.accumulate(np.r_[1, nav[method].to_numpy()])[1:]
    axes[0, 1].plot(nav.index, nav[method] / running_max - 1, label=method, color=colors[method])
axes[0, 0].set(title="NAV after transaction costs", ylabel="Starting capital = 1")
axes[0, 0].legend()
axes[0, 1].set(title="Drawdown", ylabel="Fraction of prior peak")
axes[1, 0].plot(reduction_comparison.fold, reduction_comparison.jaccard, color="#56806F")
axes[1, 0].set(title="AUR / QAUR candidate overlap", ylabel="Jaccard", xlabel="Monthly fold", ylim=(-0.02, 1.02))
for method in ("AUR", "QAUR"):
    data = main.loc[main.method.eq(method)]
    axes[1, 1].plot(data.date, data.cash_weight, label=method, color=colors[method], alpha=0.8)
axes[1, 1].set(title="Realized cash allocation", ylabel="Weight", ylim=(-0.02, 1.02))
axes[1, 1].legend()
for axis in axes.flat:
    axis.grid(alpha=0.2)
fig.savefig(RUN_DIR / "research_results.png", dpi=180)
plt.show()

## 15. Danh mục mục tiêu kỳ kế tiếp và diễn giải

Danh mục dự kiến dùng lần giá cuối bộ dữ liệu, chưa có khớp lệnh hay lợi nhuận
tương lai. Shadow weight là tỷ trọng cổ phiếu nếu mở gate; target weight là sau gate.
Cash target lặp trên mỗi hàng để dễ đọc, không cộng cột cash giữa các mã.

In [ ]:
next_basket = selections.loc[selections.seed.eq(CFG.seeds[0]) & selections.prospective & selections.selected].copy()
display(next_basket[["decision", "method", "ticker", "prediction_20d", "shadow_weight", "target_weight", "cash_target"]].round(6))
reference = performance.loc[performance.seed.eq(CFG.seeds[0]) & performance.period.eq("all_observed")].set_index("method")
interpretation = (
    f"Bản viết lại chạy {int((~calendar.prospective).sum())} tháng và {len(CFG.seeds)} seed. "
    f"Lợi nhuận tích lũy (seed {CFG.seeds[0]}): AUR {reference.loc['AUR', 'cumulative_return']:.2%}, "
    f"QAUR {reference.loc['QAUR', 'cumulative_return']:.2%}. "
    f"Số seed QAUR vượt AUR: {int(seed_robustness.positive_difference.sum())}/{len(CFG.seeds)}. "
    f"Có {int(trade_ledger.status.ne('executed').sum())} lần tái cân bằng bị chặn một phần hoặc toàn bộ lệnh. "
    f"Tổng số vị thế-ngày định giá stale: {int(portfolio_returns.stale_positions.sum())}. "
    "Các kết quả thuộc thực nghiệm triển khai lại trên dữ liệu đã biết; "
    "XY-QAOA là mô phỏng trạng thái trên CPU. H1 chịu ảnh hưởng khởi tạo QAUR bằng AUR."
)
display(Markdown(interpretation))

## 16. Audit cuối: dữ liệu, thời gian, bộ giải, giao dịch

Chỉ báo thành công khi kiểm tra trên chính kết quả vừa chạy đều đạt. Lợi nhuận
thấp hoặc giả thuyết không được ủng hộ không phải lỗi code và không bị sửa số liệu.
Các điều kiện audit tập trung vào tính nhất quán dữ liệu và thuật toán.

In [ ]:
audit_rows = []
def audit(label, condition, evidence):
    audit_rows.append({"check": label, "passed": bool(condition), "evidence": str(evidence)})

keys = ["seed", "fold", "method"]
selected = selections.loc[selections.selected]
audit("Exact dataset identity", provenance["sha256"] == CFG.expected_sha256, provenance["sha256"])
audit("All observed months forecast", len(forecast_cache) == len(calendar), len(calendar))
audit("Train labels purged before validation", (forecast_diagnostics.train_label_end < forecast_diagnostics.validation_start).all(), "strict <")
audit("Validation/refit labels purged before decision", (forecast_diagnostics.validation_label_end < forecast_diagnostics.decision).all()
      and (forecast_diagnostics.refit_label_end < forecast_diagnostics.decision).all(), "strict <")
audit("Snapshot available before order", (forecast_diagnostics.available_at_max <= forecast_diagnostics.decision).all(), "available_at <= decision")
audit("All seeds and both reducers", len(solver_results) == len(CFG.seeds) * len(calendar) * 2, len(solver_results))
audit("Top-K cardinality", selections.groupby(keys).size().eq(CFG.candidates).all(), CFG.candidates)
audit("Portfolio cardinality", selected.groupby(keys).size().eq(CFG.cardinality).all(), CFG.cardinality)
audit("Shadow weights sum to one", np.allclose(selected.groupby(keys).shadow_weight.sum(), 1, atol=1e-9), "1 before gate")
audit("Weight bounds", selected.shadow_weight.between(CFG.lower_weight - 1e-10, CFG.upper_weight + 1e-10).all(), "[0.05,0.30]")
audit("XY feasible samples", solver_results.feasibility_rate.eq(1).all(), solver_results.feasibility_rate.min())
audit("XY probability normalization", np.allclose(solver_results.probability_sum, 1), solver_results.probability_sum.min())
audit("Exact reference lower bound", solver_results.optimality_gap.ge(-1e-9).all(), solver_results.optimality_gap.min())
audit("Self financing", trade_ledger.reconciliation_error.le(1e-9).all(), trade_ledger.reconciliation_error.max())
audit("Nonnegative costs", portfolio_returns.cost.ge(0).all(), portfolio_returns.cost.sum())
audit("Baseline invests when quotes allow", main.loc[main.method.eq('UNIVERSE_EW'), 'positions'].gt(0).any(), "partial fills leave cash for unavailable names")
audit("Positive finite NAV", np.isfinite(portfolio_returns.nav).all() and portfolio_returns.nav.gt(0).all(), portfolio_returns.nav.min())
audit("Net/gross cost reconciliation", all(np.allclose(g.gross_return - g.net_return,
      g.cost / g.nav.shift(1, fill_value=1), atol=1e-10) for _, g in portfolio_returns.groupby(["seed", "method"])), "cost / previous NAV")
audit("Unique daily keys", not portfolio_returns.duplicated(["seed", "method", "date"]).any(), len(portfolio_returns))
audit("All calendar sessions simulated", portfolio_returns.groupby(["seed", "method"]).size().eq(len(DATES[DATES >= calendar.iloc[0].decision])).all(), len(wide))
audit("Cumulative NAV reconciles", all(np.allclose((1 + g.net_return).cumprod(), g.nav, atol=1e-9)
      for _, g in portfolio_returns.groupby(["seed", "method"])), "cumprod(net_return) == NAV")
audit("Gate and cash targets sum to one", np.allclose(selected.groupby(keys).target_weight.sum() + selected.groupby(keys).cash_target.first(), 1), "target + cash = 1")
audit("Holm adjustment valid", hypotheses.holm_pvalue.ge(hypotheses.pvalue - 1e-12).all() and hypotheses.holm_pvalue.between(0, 1).all(), len(hypotheses))
audit("Benchmark not extended into 2026", benchmark_comparison.end.le(pd.Timestamp('2025-12-31')).all(), str(common_dates.max()))
audit("Core algorithms checked", kernel_checks(), "QUBO / XY / weights / causal features")
audit_table = pd.DataFrame(audit_rows)
display(audit_table)
assert audit_table.passed.all(), audit_table.loc[~audit_table.passed].to_dict("records")
print(f"FROM_SCRATCH_RESEARCH_OK: {len(audit_table)}/{len(audit_table)} kiểm tra")

## 17. Xuất kết quả và tài liệu phương pháp

ZIP gồm tất cả kết quả tính mới, ledger, audit, cấu hình, môi trường, nguồn gốc
dữ liệu và biểu đồ. Tệp gốc không bị thay đổi. Link Download hiển thị trên Colab;
chọn RUN_DOWNLOAD = True để tải qua hộp thoại của trình duyệt.

Nguồn phương pháp: [XGBoost API](https://xgboost.readthedocs.io/en/stable/python/python_api.html),
[Hadfield và cộng sự, alternating operator ansatz](https://arxiv.org/abs/1709.03489).
Các URL này chỉ là tài liệu tham khảo; không được gọi để lấy mã khi thực thi.

In [ ]:
import shutil
from IPython.display import FileLink

tables = {"input_audit": data_audit, "fold_calendar": calendar, "forecast_diagnostics": forecast_diagnostics,
          "forecast_snapshots": forecast_snapshots.drop(columns=["target", "label_end"], errors="ignore"),
          "selections": selections, "solver_diagnostics": solver_results, "portfolio_returns": portfolio_returns,
          "trade_ledger": trade_ledger, "performance": performance, "benchmark_common_period": benchmark_comparison,
          "reducer_comparison": reduction_comparison, "hypotheses": hypotheses,
          "seed_robustness": seed_robustness.reset_index(), "next_basket": next_basket, "audit": audit_table}
for name, frame in tables.items():
    frame.to_csv(RUN_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
run_manifest = {"implementation": "independent_cell_by_cell_rebuild", "dataset": provenance,
                "config": asdict(CFG), "environment": {p: importlib.metadata.version(p) for p in requirements},
                "python": sys.version, "months": int((~calendar.prospective).sum()),
                "seeds": list(CFG.seeds), "audit_checks": len(audit_table), "all_audits_passed": bool(audit_table.passed.all()),
                "qa_reducer": "classical simulated annealing on fixed-cardinality QUBO",
                "portfolio_solver": "XY-QAOA simulated in feasible subspace; exact enumeration for gap only",
                "evidence": "retrospective_reimplementation_on_previously_observed_data",
                "missing_price_policy": "last observed valuation; freeze untradeable positions; partial fills; record stale positions",
                "elapsed_seconds": time.perf_counter() - STARTED,
                "files": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in RUN_DIR.iterdir() if p.is_file()}}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "interpretation.txt").write_text(interpretation, encoding="utf-8")
RESULT_ZIP = Path(shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR))
assert RESULT_ZIP.is_file()
print("Kết quả:", RESULT_ZIP.name, "; thời gian:", round(run_manifest["elapsed_seconds"], 1), "giây")
display(FileLink(str(RESULT_ZIP)))
RUN_DOWNLOAD = False
if RUN_DOWNLOAD:
    from google.colab import files
    files.download(str(RESULT_ZIP))